# Mitsubishi Stock Management — Analytical Data Preparation

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Inspect current headers

In [4]:
df = pd.read_csv("merged_output.csv")
item = pd.read_excel("Item.xlsx")
customer = pd.read_excel("Customer.xlsx")

In [5]:
print("df columns:")
print(df.columns.tolist())

print("\nitem columns:")
print(item.columns.tolist())

print("\ncustomer columns:")
print(customer.columns.tolist())

df columns:
['order_key', 'sales_order_date', 'sales_order_status', 'customer_internal_id', 'company_name', 'item_code', 'base_price', 'quantity', 'customer_type', 'fa_sub_industries', 'item_internal_id', 'production_status', 'product_category_1', 'product_category_2', 'list_price']

item columns:
[' Item Internal ID', 'Item Code', 'Production Status', 'Product Category 1', 'Product Category 2', 'List price']

customer columns:
['Customer Internal ID ', 'Company Name', 'Customer Type', 'FA Sub Industries']


## 2. Prepare transaction data

In [6]:
df["sales_order_date"] = pd.to_datetime(df["sales_order_date"])
df = df.sort_values("sales_order_date").copy()

if "gross_revenue" not in df.columns:
    df["gross_revenue"] = df["list_price"] * df["quantity"]

if "net_revenue" not in df.columns:
    df["net_revenue"] = df["base_price"] * df["quantity"]

if "discount_amount" not in df.columns:
    df["discount_amount"] = (
        df["list_price"] - df["base_price"]
    ) * df["quantity"]

if "discount_pct" not in df.columns:
    df["discount_pct"] = (
        df["discount_amount"] /
        df["gross_revenue"].replace(0, np.nan)
    ).fillna(0)

## 3. Customer × Product × Order table

In [7]:
customer_product_orders = (
    df.groupby([
        "customer_internal_id",
        "item_internal_id",
        "item_code",
        "order_key",
        "sales_order_date"
    ], as_index=False)
    .agg(
        order_quantity=("quantity", "sum"),
        order_revenue=("net_revenue", "sum")
    )
)

customer_product_orders = customer_product_orders.sort_values([
    "customer_internal_id",
    "item_internal_id",
    "sales_order_date",
    "order_key"
]).copy()

customer_product_orders["previous_order_date"] = (
    customer_product_orders
    .groupby(["customer_internal_id", "item_internal_id"])
    ["sales_order_date"].shift(1)
)

customer_product_orders["reorder_interval_days"] = (
    customer_product_orders["sales_order_date"] -
    customer_product_orders["previous_order_date"]
).dt.days

customer_product_orders["reorder_number"] = (
    customer_product_orders
    .groupby(["customer_internal_id", "item_internal_id"])
    .cumcount()
)

## 4. Customer × Product reorder analytics

In [8]:
customer_product_reorder = (
    customer_product_orders
    .groupby([
        "customer_internal_id",
        "item_internal_id",
        "item_code"
    ], as_index=False)
    .agg(
        first_order_date=("sales_order_date", "min"),
        last_order_date=("sales_order_date", "max"),
        order_count=("order_key", "nunique"),
        total_quantity=("order_quantity", "sum"),
        avg_order_quantity=("order_quantity", "mean"),
        avg_reorder_interval_days=("reorder_interval_days", "mean"),
        median_reorder_interval_days=("reorder_interval_days", "median"),
        reorder_interval_std=("reorder_interval_days", "std")
    )
)

analysis_date = df["sales_order_date"].max()

customer_product_reorder["reorder_interval_count"] = (
    customer_product_reorder["order_count"] - 1
)

customer_product_reorder["days_since_last_order"] = (
    analysis_date -
    customer_product_reorder["last_order_date"]
).dt.days

customer_product_reorder["reorder_interval_cv"] = (
    customer_product_reorder["reorder_interval_std"] /
    customer_product_reorder["avg_reorder_interval_days"]
)

In [9]:
def reorder_predictability(cv):
    if pd.isna(cv):
        return "Insufficient History"
    elif cv <= 0.25:
        return "Highly Predictable"
    elif cv <= 0.50:
        return "Moderately Predictable"
    else:
        return "Unpredictable"

def reorder_confidence(n):
    if n < 2:
        return "Insufficient"
    elif n < 4:
        return "Low"
    elif n < 7:
        return "Medium"
    return "High"

customer_product_reorder["reorder_predictability"] = (
    customer_product_reorder["reorder_interval_cv"]
    .apply(reorder_predictability)
)

customer_product_reorder["reorder_confidence"] = (
    customer_product_reorder["reorder_interval_count"]
    .apply(reorder_confidence)
)

customer_product_reorder["expected_reorder_date"] = (
    customer_product_reorder["last_order_date"] +
    pd.to_timedelta(
        customer_product_reorder["median_reorder_interval_days"],
        unit="D"
    )
)

customer_product_reorder["reorder_progress_pct"] = (
    customer_product_reorder["days_since_last_order"] /
    customer_product_reorder["median_reorder_interval_days"].replace(0, np.nan)
) * 100

In [10]:
conditions = [
    customer_product_reorder["reorder_interval_count"] < 2,
    customer_product_reorder["reorder_progress_pct"] < 70,
    customer_product_reorder["reorder_progress_pct"].between(70, 99.99),
    customer_product_reorder["reorder_progress_pct"].between(100, 129.99),
    customer_product_reorder["reorder_progress_pct"] >= 130
]

choices = [
    "Insufficient History",
    "Not Due",
    "Approaching Reorder",
    "Expected Now",
    "Overdue"
]

customer_product_reorder["reorder_status"] = np.select(
    conditions,
    choices,
    default="Insufficient History"
)

## 5. Product-level analytics for `item`

In [11]:
item_sales = (
    df.groupby("item_internal_id", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_revenue=("net_revenue", "sum"),
        total_orders=("order_key", "nunique"),
        unique_customers=("customer_internal_id", "nunique"),
        first_sale_date=("sales_order_date", "min"),
        last_sale_date=("sales_order_date", "max")
    )
)

date_range_days = (
    df["sales_order_date"].max() -
    df["sales_order_date"].min()
).days + 1

item_sales["avg_units_per_day"] = (
    item_sales["total_quantity"] / date_range_days
)

item_sales["avg_units_per_month"] = (
    item_sales["avg_units_per_day"] * 30.4375
)

### Product monthly demand and XYZ

In [12]:
df["year_month"] = (
    df["sales_order_date"].dt.to_period("M").astype(str)
)

product_monthly = (
    df.groupby(["item_internal_id", "year_month"], as_index=False)
    .agg(
        quantity=("quantity", "sum"),
        revenue=("net_revenue", "sum")
    )
)

item_demand_stats = (
    product_monthly
    .groupby("item_internal_id")
    .agg(
        monthly_demand_mean=("quantity", "mean"),
        monthly_demand_std=("quantity", "std")
    )
    .reset_index()
)

item_demand_stats["demand_cv"] = (
    item_demand_stats["monthly_demand_std"] /
    item_demand_stats["monthly_demand_mean"]
)

def xyz_class(cv):
    if pd.isna(cv):
        return "Insufficient Data"
    elif cv <= 0.5:
        return "X"
    elif cv <= 1.0:
        return "Y"
    return "Z"

item_demand_stats["xyz_class"] = (
    item_demand_stats["demand_cv"].apply(xyz_class)
)

### ABC classification

In [13]:
item_revenue = (
    df.groupby("item_internal_id")["net_revenue"]
    .sum()
    .sort_values(ascending=False)
)

item_revenue_share = item_revenue / item_revenue.sum()
item_cumulative_share = item_revenue_share.cumsum()

def abc_class(x):
    if x <= 0.80:
        return "A"
    elif x <= 0.95:
        return "B"
    return "C"

item_abc = pd.DataFrame({
    "item_internal_id": item_cumulative_share.index,
    "revenue_share": item_revenue_share.values,
    "cumulative_revenue_share": item_cumulative_share.values
})

item_abc["abc_class"] = (
    item_abc["cumulative_revenue_share"].apply(abc_class)
)

### Product reorder summary

In [14]:
item_reorder = (
    customer_product_reorder
    .groupby("item_internal_id", as_index=False)
    .agg(
        reorder_customers=("customer_internal_id", "nunique"),
        avg_reorder_interval_days=("avg_reorder_interval_days", "mean"),
        median_reorder_interval_days=("median_reorder_interval_days", "median"),
        reorder_interval_std=("reorder_interval_std", "mean"),
        customers_approaching_reorder=(
            "reorder_status",
            lambda x: (x == "Approaching Reorder").sum()
        ),
        customers_expected_now=(
            "reorder_status",
            lambda x: (x == "Expected Now").sum()
        ),
        customers_overdue=(
            "reorder_status",
            lambda x: (x == "Overdue").sum()
        )
    )
)

### Merge product analytics into `item`

In [17]:
item.rename(columns=lambda x: x.strip(), inplace=True)

In [18]:
item = (
    item
    .merge(item_sales, left_on="Item Internal ID",
           right_on="item_internal_id", how="left")
    .merge(item_demand_stats, left_on="Item Internal ID",
           right_on="item_internal_id", how="left",
           suffixes=("", "_demand"))
    .merge(item_abc, left_on="Item Internal ID",
           right_on="item_internal_id", how="left",
           suffixes=("", "_abc"))
    .merge(item_reorder, left_on="Item Internal ID",
           right_on="item_internal_id", how="left",
           suffixes=("", "_reorder"))
)

helper_cols = [
    c for c in item.columns
    if c.startswith("item_internal_id")
    and c != "Item Internal ID"
]
item = item.drop(columns=helper_cols, errors="ignore")

item["is_discontinued"] = (
    item["Production Status"] == "Discontinue"
).astype(int)

item["is_active_product"] = (
    item["Production Status"] == "Continue"
).astype(int)

item["discontinued_demand_risk"] = np.where(
    (item["is_discontinued"] == 1) &
    (item["total_quantity"].fillna(0) > 0),
    "Review",
    "Normal"
)

## 6. Customer-level analytics for `customer`

In [19]:
customer_sales = (
    df.groupby("customer_internal_id", as_index=False)
    .agg(
        total_revenue=("net_revenue", "sum"),
        total_quantity=("quantity", "sum"),
        total_orders=("order_key", "nunique"),
        unique_products=("item_internal_id", "nunique"),
        first_order_date=("sales_order_date", "min"),
        last_order_date=("sales_order_date", "max")
    )
)

customer_sales["average_order_value"] = (
    customer_sales["total_revenue"] /
    customer_sales["total_orders"].replace(0, np.nan)
)

customer_sales["recency_days"] = (
    analysis_date - customer_sales["last_order_date"]
).dt.days

customer_sales["customer_lifetime_days"] = (
    customer_sales["last_order_date"] -
    customer_sales["first_order_date"]
).dt.days

### Customer reorder behavior

In [20]:
customer_reorder = (
    customer_product_reorder
    .groupby("customer_internal_id", as_index=False)
    .agg(
        products_reordered=("item_internal_id", "nunique"),
        avg_reorder_interval_days=("avg_reorder_interval_days", "mean"),
        median_reorder_interval_days=("median_reorder_interval_days", "median"),
        reorderable_products=(
            "reorder_interval_count",
            lambda x: (x >= 2).sum()
        ),
        products_approaching_reorder=(
            "reorder_status",
            lambda x: (x == "Approaching Reorder").sum()
        ),
        products_expected_now=(
            "reorder_status",
            lambda x: (x == "Expected Now").sum()
        ),
        products_overdue=(
            "reorder_status",
            lambda x: (x == "Overdue").sum()
        )
    )
)

### RFM raw metrics

In [21]:
rfm = customer_sales[
    ["customer_internal_id", "recency_days",
     "total_orders", "total_revenue"]
].copy()

rfm = rfm.rename(columns={
    "total_orders": "frequency",
    "total_revenue": "monetary"
})

rfm["r_score"] = pd.qcut(
    rfm["recency_days"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["f_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["m_score"] = pd.qcut(
    rfm["monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

### Merge customer analytics into `customer`

In [23]:
customer.rename(columns=lambda x: x.strip(), inplace=True)

In [24]:
customer = (
    customer
    .merge(customer_sales,
           left_on="Customer Internal ID",
           right_on="customer_internal_id",
           how="left")
    .merge(customer_reorder,
           left_on="Customer Internal ID",
           right_on="customer_internal_id",
           how="left",
           suffixes=("", "_reorder"))
    .merge(rfm,
           left_on="Customer Internal ID",
           right_on="customer_internal_id",
           how="left",
           suffixes=("", "_rfm"))
)

helper_cols = [
    c for c in customer.columns
    if c.startswith("customer_internal_id")
    and c != "Customer Internal ID"
]
customer = customer.drop(columns=helper_cols, errors="ignore")

## 7. Validation

In [25]:
print("Duplicate items:", item["Item Internal ID"].duplicated().sum())
print("Duplicate customers:", customer["Customer Internal ID"].duplicated().sum())

print(
    "Items with sales data:",
    item["total_quantity"].notna().sum(),
    "/", len(item)
)

print(
    "Customers with sales data:",
    customer["total_revenue"].notna().sum(),
    "/", len(customer)
)

print("Customer × Product reorder rows:", len(customer_product_reorder))
print("Item shape:", item.shape)
print("Customer shape:", customer.shape)
print("Transaction shape:", df.shape)

Duplicate items: 0
Duplicate customers: 0
Items with sales data: 5706 / 5706
Customers with sales data: 2159 / 2159
Customer × Product reorder rows: 50959
Item shape: (5706, 31)
Customer shape: (2159, 26)
Transaction shape: (265135, 20)


## 8. Inspect resulting tables

In [26]:
display(item.head())
display(customer.head())
display(customer_product_reorder.head())

,Item Internal ID,Item Code,Production Status,Product Category 1,Product Category 2,List price,total_quantity,total_revenue,total_orders,unique_customers,first_sale_date,last_sale_date,avg_units_per_day,avg_units_per_month,monthly_demand_mean,monthly_demand_std,demand_cv,xyz_class,revenue_share,cumulative_revenue_share,abc_class,reorder_customers,avg_reorder_interval_days,median_reorder_interval_days,reorder_interval_std,customers_approaching_reorder,customers_expected_now,customers_overdue,is_discontinued,is_active_product,discontinued_demand_risk
0,295356,AAA,Continue,Geared Motor,Geared Motor,1026.0,1,41026.00,1,1,2023-04-10,2023-04-10,0.000684,0.020833,1.000000,NaN,NaN,Insufficient Data,0.000004,0.997290,C,1,NaN,NaN,NaN,0,0,0,0,1,Normal
1,99047,AAB,Continue,Robot,Robot Others,13421.0,2,23116.24,2,1,2021-09-21,2024-11-18,0.001369,0.041667,1.000000,0.000000,0.000000,X,0.000002,0.998884,C,1,1154.0,1154.0,NaN,0,0,0,0,1,Normal
2,24903,AAC,Continue,Robot,Robot Others,15745.0,6,57805.00,4,3,2021-10-01,2024-11-04,0.004107,0.125000,1.500000,0.577350,0.384900,X,0.000005,0.995817,C,3,146.0,146.0,NaN,0,0,0,0,1,Normal
3,96823,AAD,Continue,Robot,Robot Others,15745.0,7,88630.15,3,1,2021-09-09,2022-11-25,0.004791,0.145833,2.333333,2.309401,0.989743,Y,0.000008,0.992975,C,1,221.0,221.0,169.705627,0,0,1,0,1,Normal
4,192156,AAE,Continue,Robot,Robot Others,65796.0,1,32315.88,1,1,2021-04-26,2021-04-26,0.000684,0.020833,1.000000,NaN,NaN,Insufficient Data,0.000003,0.998085,C,1,NaN,NaN,NaN,0,0,0,0,1,Normal


,Customer Internal ID,Company Name,Customer Type,FA Sub Industries,total_revenue,total_quantity,total_orders,unique_products,first_order_date,last_order_date,average_order_value,recency_days,customer_lifetime_days,products_reordered,avg_reorder_interval_days,median_reorder_interval_days,reorderable_products,products_approaching_reorder,products_expected_now,products_overdue,recency_days_rfm,frequency,monetary,r_score,f_score,m_score
0,4620,"AAA CO.,LTD.",VIP,Wholesale and Distribution,1.153893e+09,105227,13717,2236,2021-04-01,2025-03-31,84121.368550,0,1460,2236,150.908552,53.00,1376,72,98,834,0,13717,1.153893e+09,5,5,5
1,4693,"AAB CO.,LTD.",VIP,Wholesale and Distribution,5.166573e+08,32152,6048,1648,2021-04-02,2025-03-31,85426.142707,0,1459,1648,196.069606,90.00,818,81,53,346,0,6048,5.166573e+08,5,5,5
2,4705,"AAC CO.,LTD.",VIP,Wholesale and Distribution,5.923102e+08,54599,7843,1450,2021-04-01,2025-03-31,75520.878030,0,1460,1450,174.658099,58.75,867,58,38,590,0,7843,5.923102e+08,5,5,5
3,4743,"AAD CO.,LTD.",VIP,Wholesale and Distribution,8.442419e+08,65323,14081,1515,2021-04-01,2025-03-31,59956.100583,0,1460,1515,161.971209,61.50,950,100,62,383,0,14081,8.442419e+08,5,5,5
4,5078,"AAE CO.,LTD.",VIP,Wholesale and Distribution,3.253429e+08,20491,2636,1143,2021-04-01,2025-03-31,123422.953190,0,1460,1143,190.792379,79.50,600,48,36,368,0,2636,3.253429e+08,5,5,5


,customer_internal_id,item_internal_id,item_code,first_order_date,last_order_date,order_count,total_quantity,avg_order_quantity,avg_reorder_interval_days,median_reorder_interval_days,reorder_interval_std,reorder_interval_count,days_since_last_order,reorder_interval_cv,reorder_predictability,reorder_confidence,expected_reorder_date,reorder_progress_pct,reorder_status
0,702,16690,IBS,2023-08-07,2023-08-07,1,1,1.0,NaN,NaN,NaN,0,602,NaN,Insufficient History,Insufficient,NaT,NaN,Insufficient History
1,702,57209,ICD,2022-12-01,2022-12-01,1,1,1.0,NaN,NaN,NaN,0,851,NaN,Insufficient History,Insufficient,NaT,NaN,Insufficient History
2,702,134587,ICE,2022-12-01,2022-12-01,1,5,5.0,NaN,NaN,NaN,0,851,NaN,Insufficient History,Insufficient,NaT,NaN,Insufficient History
3,703,7784,ASF,2021-08-09,2021-08-09,1,2,2.0,NaN,NaN,NaN,0,1330,NaN,Insufficient History,Insufficient,NaT,NaN,Insufficient History
4,703,8100,BXQ,2021-05-12,2021-05-12,1,2,2.0,NaN,NaN,NaN,0,1419,NaN,Insufficient History,Insufficient,NaT,NaN,Insufficient History


## 9. Export

In [29]:
item.to_excel("output/Item_Analytical.xlsx", index=False)
customer.to_excel("output/Customer_Analytical.xlsx", index=False)
customer_product_reorder.to_excel("output/Customer_Product_Reorder.xlsx", index=False)
df.to_parquet("output/Fact_Sales_Analytical.parquet", index=False)

In [30]:
df.to_csv("output/Fact_Sales_Analytical.csv", index=False)

## 10. Create product CoOrdering Insight table

this idea came later so it was added at the end

In [ ]:


# ---------------------------------------------------------
# 1. Create unique product lists for each order
# ---------------------------------------------------------

order_products = (
    df.groupby("order_key")["item_internal_id"]
    .apply(lambda x: sorted(set(x.dropna())))
)

# Remove orders containing only one unique product
order_products = order_products[
    order_products.apply(len) >= 2
]


# ---------------------------------------------------------
# 2. Count individual product orders
# ---------------------------------------------------------

product_order_count = (
    df.groupby("item_internal_id")["order_key"]
    .nunique()
    .to_dict()
)

total_orders = df["order_key"].nunique()


# ---------------------------------------------------------
# 3. Generate product pairs
# ---------------------------------------------------------

pair_counts = {}

for products in order_products:
    for product_a, product_b in combinations(products, 2):

        pair = (product_a, product_b)

        pair_counts[pair] = pair_counts.get(pair, 0) + 1


# ---------------------------------------------------------
# 4. Convert pairs into DataFrame
# ---------------------------------------------------------

co_ordering = pd.DataFrame(
    [
        {
            "item_a": pair[0],
            "item_b": pair[1],
            "orders_together": count
        }
        for pair, count in pair_counts.items()
    ]
)


# ---------------------------------------------------------
# 5. Calculate association metrics
# ---------------------------------------------------------

co_ordering["item_a_order_count"] = (
    co_ordering["item_a"]
    .map(product_order_count)
)

co_ordering["item_b_order_count"] = (
    co_ordering["item_b"]
    .map(product_order_count)
)


# Support
# Percentage of all orders containing both products

co_ordering["support"] = (
    co_ordering["orders_together"] /
    total_orders
)


# Confidence A -> B
# When A is ordered, how often is B also ordered?

co_ordering["confidence_a_to_b"] = (
    co_ordering["orders_together"] /
    co_ordering["item_a_order_count"]
)


# Confidence B -> A
# When B is ordered, how often is A also ordered?

co_ordering["confidence_b_to_a"] = (
    co_ordering["orders_together"] /
    co_ordering["item_b_order_count"]
)


# ---------------------------------------------------------
# 6. Lift
# ---------------------------------------------------------
#
# Lift > 1  = products occur together more often than expected
# Lift = 1  = no meaningful association
# Lift < 1  = products occur together less often than expected
#

prob_a = co_ordering["item_a_order_count"] / total_orders
prob_b = co_ordering["item_b_order_count"] / total_orders

co_ordering["lift"] = (
    co_ordering["support"] /
    (prob_a * prob_b)
)


# ---------------------------------------------------------
# 7. Average number of items in the order
# ---------------------------------------------------------

order_size = (
    df.groupby("order_key")["item_internal_id"]
    .nunique()
)

co_ordering["avg_order_product_count"] = (
    order_size.mean()
)


# ---------------------------------------------------------
# 8. Make percentages easier to read
# ---------------------------------------------------------

co_ordering["support_pct"] = (
    co_ordering["support"] * 100
)

co_ordering["confidence_a_to_b_pct"] = (
    co_ordering["confidence_a_to_b"] * 100
)

co_ordering["confidence_b_to_a_pct"] = (
    co_ordering["confidence_b_to_a"] * 100
)


# ---------------------------------------------------------
# 9. Sort by strongest association
# ---------------------------------------------------------

co_ordering = co_ordering.sort_values(
    ["lift", "orders_together"],
    ascending=[False, False]
).reset_index(drop=True)


# ---------------------------------------------------------
# 10. Export to Excel
# ---------------------------------------------------------

co_ordering.to_excel(
    "Product_CoOrdering_Analysis.xlsx",
    index=False
)

print(
    f"Created Product_CoOrdering_Analysis.xlsx "
    f"with {len(co_ordering):,} product pairs."
)

Created Product_CoOrdering_Analysis.xlsx with 242,025 product pairs.
